# Jersey-number localisation: full training run on Kaggle

This notebook reproduces the experiment in the repository
[MHKhan50/jersey-number-detection](https://github.com/MHKhan50/jersey-number-detection):

1. dataset statistics and a cross-split duplicate check,
2. two baselines (mean box, and the v1 CNN regressor),
3. JerseyNet, an anchor-free detector trained from scratch, over three seeds,
4. COCO-style AP and the v1 first-box IoU on validation and test, with 95% bootstrap intervals.

**Kaggle settings:** Accelerator *GPU T4 x2* (the P100 is not supported by recent PyTorch builds),
Internet *on*, dataset attached under *Input* (a folder with `train/`, `valid/`, `test/`) or a
`ROBOFLOW_API_KEY` secret. Then *Save Version → Save & Run All (Commit)*.

Everything is written to `/kaggle/working/results`. Training resumes from `last.pt` if the same
output folder is reused, and every stage writes `summary.json`, so a late failure does not lose
earlier results.

In [ ]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"                    # branch or tag to train from
CONFIG = "configs/kaggle.json"       # seeds, epochs, image size, ...
TIME_BUDGET_HOURS = 10.5             # Kaggle stops GPU sessions at 12 h
OUT_DIR = "/kaggle/working/results"
INPUT_ROOTS = ["/kaggle/input"]
DOWNLOAD_DIR = "/tmp/jersey-2"

# Continuous-integration smoke test only: a tiny synthetic dataset and two epochs.
SMOKE = os.environ.get("JND_SMOKE") == "1"
if SMOKE:
    CONFIG, TIME_BUDGET_HOURS = "configs/smoke.json", 0.3
    OUT_DIR = os.environ.get("JND_OUT", "/tmp/jnd-smoke/results")
print({"REPO_REF": REPO_REF, "CONFIG": CONFIG, "TIME_BUDGET_HOURS": TIME_BUDGET_HOURS, "SMOKE": SMOKE})

In [ ]:
# Environment check
import subprocess

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
    major, minor = torch.cuda.get_device_capability(0)
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("WARNING: this PyTorch build has no kernels for this GPU. "
              "Switch the accelerator to GPU T4 x2. Training would fall back to the CPU.")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2 in the notebook settings.")

In [ ]:
# Get the code (pinned commit is recorded in results/environment.json)
import sys
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
        if r.returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
import jersey_detect

print("jersey_detect", jersey_detect.__version__, "from", SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Find the dataset (Kaggle input first, then Roboflow with the ROBOFLOW_API_KEY secret)
from jersey_detect.prepare import locate_or_download

DATA_ROOT = None
try:
    if SMOKE:
        from jersey_detect.synthetic import make_dataset
        DATA_ROOT = make_dataset("/tmp/jnd-smoke/data")
    else:
        DATA_ROOT = locate_or_download(INPUT_ROOTS, DOWNLOAD_DIR)
    print("Dataset:", DATA_ROOT)
    for split in ("train", "valid", "val", "test"):
        p = Path(DATA_ROOT) / split / "images"
        if p.is_dir():
            print(f"  {split}: {sum(1 for _ in p.iterdir())} images")
except Exception as exc:
    print("DATASET NOT FOUND:", exc)
    print("Inputs attached:", [p.name for p in Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else "none")
    # Stop here so the version is marked as failed instead of finishing without training.
    raise RuntimeError("No dataset: attach jersey-2 as an input or the ROBOFLOW_API_KEY secret") from exc

In [ ]:
# Run the whole experiment in a separate process, so a crash cannot take the notebook down.
# Output streams here and is also saved to results/run.log.
return_code = None
if DATA_ROOT is not None:
    env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-m", "jersey_detect", "run", "--data", str(DATA_ROOT),
           "--out", OUT_DIR, "--config", str(REPO_DIR / CONFIG), "--hours", str(TIME_BUDGET_HOURS)]
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return_code = proc.wait()
print("return code:", return_code)

In [ ]:
# Results
import json

summary_path = Path(OUT_DIR) / "summary.json"
if summary_path.exists():
    print((Path(OUT_DIR) / "results.md").read_text())
    summary = json.loads(summary_path.read_text())
    print("Stages:", json.dumps(summary["stages"], indent=1))
    if summary["errors"]:
        print("Errors:", json.dumps(summary["errors"], indent=1))
    for p in sorted(Path(OUT_DIR).rglob("*")):
        if p.is_file():
            print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT_DIR)}")
else:
    print("No summary written; see the output of the previous cells.")